In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras import metrics

def model_training_25d(x: np.ndarray, labels: np.ndarray):
    '''
    Takes in a series of ndarrays and trains a model using a 3D CNN architecture.
    '''
    # Input layer: Shape is (height, width, channels)
    input_tensor = layers.Input(shape=(1, 224, 224, 24))

    ### First Convolution & MaxPooling
    net = layers.Conv3D(32, 3, padding='same', activation='relu')(input_tensor)
    net = layers.MaxPooling3D(pool_size=(1, 2, 2), padding='same')(net)

    ### Second Convolution & MaxPooling
    net = layers.Conv3D(64, 3, padding='same', activation='relu')(net)
    net = layers.MaxPooling3D(pool_size=(3, 3, 3), padding='same')(net)

    ### Third Convolution & MaxPooling
    net = layers.Conv3D(64, 3, padding='same', activation='relu')(net)
    net = layers.MaxPooling3D(pool_size=(2, 2, 2), padding='same')(net)

    ### Flattening
    shared_feature = layers.GlobalAveragePooling()(net)

    ### Fully Connected Layers
    net = layers.Dense(20, activation='relu')(shared_feature)
    net = layers.Dense(20, activation='relu')(net)

    ### Classification Layer: 12 independent binary classifications
    # One unit per condition (ACL, MCL, Meniscus, OA, Effusion, etc.)
    output_layer = layers.Dense(12, activation='sigmoid')(net)

    # Instantiate the Functional Model
    model = models.Model(inputs=input_tensor, outputs=output_layer)

    ### Compile the model
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision')
        ]
    )

    return model


In [31]:
def model_fit(model, x, y):
    '''
    Fits a model
    '''
    history = model.fit(x, y, epochs=1)
    return history

In [32]:
# --- SIMULATED DATA FOR 2 PATIENTS ---
# Patient 1
X_train = np.random.rand(1, 224, 224, 24)
y_train = np.array([[1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]]) # 1 row of 12

# --- RUNNING THE MODEL ---
# Now x sizes (2) and y sizes (2) match perfectly!
initialized_model = model_training_25d(X_train, y_train)
initialized_model

<Functional name=functional_4, built=True>

In [26]:
X_train.shape

(1, 224, 224, 24)

In [27]:
y_train.shape

(1, 12)